# Next-Word Prediction with LSTMs — NumPy vs TensorFlow vs PyTorch

This notebook trains a stacked (multi-layer) **LSTM** three ways and compares
them on the **same data, architecture, and hyper-parameters**:

1. **Manual (NumPy)** — implemented from scratch in
   [`lstm_scratch.py`](lstm_scratch.py).
2. **TensorFlow / Keras** — [`lstm_tensorflow.py`](lstm_tensorflow.py).
3. **PyTorch** — [`lstm_pytorch.py`](lstm_pytorch.py).

All three expose the same interface (`train()`, `predict()`), so the shared
helpers in [`utils.py`](utils.py) — `evaluate`, `generate` — and
[`compare.py`](compare.py)'s `compare_models` work on every model unchanged.

The task is **next-word prediction**: each input token is a dense **word
embedding**, the target is the next word (one-hot over the vocabulary), so the
model runs in `task="classification"`.

**Pipeline:** raw text → vocabulary → training sequences → train/test split →
train the three models → compare train/test accuracy → generate text.

| Section | Contents |
|---------|----------|
| 1 | Imports |
| 2 | Data |
| 3 | Next-word prediction — Word2Vec · GloVe · FastText · BERT (each a 3-way comparison) |

> **Note on the comparison.** The frameworks train with **Adam** while the
> from-scratch model uses plain **SGD + gradient clipping**, so their learned
> weights — and accuracies — differ. This is a realistic "library vs scratch"
> comparison, not a bit-for-bit match.

## 1. Imports

In [175]:
import importlib

import numpy as np

import  utils, compare, lstm_pytorch, lstm_tensorflow, lstm_scratch

# Reload local modules so edits to the .py files are picked up without
# having to restart the kernel.
for _m in ( utils, compare, lstm_pytorch, lstm_tensorflow, lstm_scratch):
    importlib.reload(_m)
from lstm_scratch import LSTM
from lstm_pytorch import PyTorchLSTM
from lstm_tensorflow import KerasLSTM
from utils import generate_dataset, train_test_split, evaluate, predict_next, generate
from compare import compare_models

In [176]:
sentences = [
'Machine Learning (ML) is a branch of Artificial Intelligence (AI)',
'that enables computers to learn from data and improve their performance',
'without being explicitly programmed. ML algorithms identify patterns in',
'historical data and use those patterns to make predictions or decisions on new data.',
'Machine learning is widely used in various applications such as recommendation systems,',
'image recognition, speech processing, fraud detection, healthcare diagnostics,',
'and autonomous vehicles. The main types of machine learning are supervised learning,',
'unsupervised learning, and reinforcement learning. As the amount of available data continues to grow,',
'machine learning plays an increasingly important role in helping organizations automate tasks,',
'gain insights, and make data-driven decisions.'
]
sentence_words = [s.split() for s in sentences]
words = [w for s in sentence_words for w in s]

## 2. Data

A small block of text to learn from. With a corpus this size the model can only
learn surface statistics, but it is enough to demonstrate and compare the LSTMs.

In [177]:
def train_models(X_train, Y_train, hidden_layers, lr, epochs, batch_size, task):
    # lr is a 3-tuple (lr_manual, lr_pytorch, lr_keras). The from-scratch model
    # uses plain SGD, so its raw gradients are small and it needs a much larger
    # learning rate than the Adam-based frameworks (which normalise by gradient
    # RMS and stay effective at ~0.01). Passing one lr per model keeps the
    # comparison fair instead of leaving the manual LSTM essentially frozen.
    lr_manual, lr_torch, lr_keras = lr

    print('-'*100)
    print('Training Manual LSTM')
    manual_lstm = LSTM(X_train, Y_train, hidden_layers=hidden_layers,
                     learning_rate=lr_manual, epochs=epochs, batch_size=batch_size, task=task)
    manual_lstm.train()

    print()
    print('-'*100)
    print('Training PyTorch LSTM')
    torch_lstm = PyTorchLSTM(X_train, Y_train, hidden_layers=hidden_layers,
                         learning_rate=lr_torch, epochs=epochs, batch_size=batch_size, task=task).train()
    
    print()
    print('-'*100)
    print('Training TensorFlow LSTM')
    keras_lstm = KerasLSTM(X_train, Y_train, hidden_layers=hidden_layers,
                         learning_rate=lr_keras, epochs=epochs, batch_size=batch_size, task=task).train()

    print()
    print('-'*100)

    trained_models = {"manual": manual_lstm,
                      "pytorch": torch_lstm,
                      'tensorflow' : keras_lstm
                      }

    return trained_models

## 3. Next-Word Prediction

Each input token is a dense **word embedding**; targets are one-hot over the
vocabulary, so the model runs in `task="classification"` and predicts the next
word.

Each subsection feeds the same corpus through a different embedding
(Word2Vec · GloVe · FastText · BERT) and runs the **full 3-way comparison**
(manual / TensorFlow / PyTorch).

### 3.1 Word2Vec embeddings

Train a Word2Vec model on our sentences (skip-gram) and use its vectors as the LSTM's input features. This section also demonstrates a **2-layer stack** (`hidden_layers=(100, 64)`).

In [178]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentence_words,
    vector_size=100,  # dimensionality of the word vectors
    window=5,        # context window size
    min_count=1,     # minimum word frequency to include in the model
    workers=4,       # number of CPU cores to use for training
    sg=1             # use skip-gram architecture (sg=0 for CBOW)
)
word_vectors = w2v_model.wv

In [179]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, word_vectors=word_vectors)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 words, with a vocabulary of 88 unique words.
X: (102, 5, 100)   Y: (102, 5, 88)   (m, T_x, vector_size)
Split 102 sequences -> 82 train / 20 test.


In [180]:
hidden_layers, lr, epochs, batch_size, task = (100,), (8.0, 0.03, 0.03), 15, 32, 'classification'
print('Word2Vec Models')
w2v_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

Word2Vec Models
----------------------------------------------------------------------------------------------------
Training Manual LSTM
epoch 1/15 - loss: 4.4724
epoch 2/15 - loss: 4.3936
epoch 3/15 - loss: 4.3858
epoch 4/15 - loss: 4.3848
epoch 5/15 - loss: 4.3709
epoch 6/15 - loss: 4.3682
epoch 7/15 - loss: 4.3676
epoch 8/15 - loss: 4.3671
epoch 9/15 - loss: 4.3659
epoch 10/15 - loss: 4.3593
epoch 11/15 - loss: 4.3633
epoch 12/15 - loss: 4.3700
epoch 13/15 - loss: 4.3621
epoch 14/15 - loss: 4.3534
epoch 15/15 - loss: 4.3732

----------------------------------------------------------------------------------------------------
Training PyTorch LSTM
[PyTorch] epoch 1/15, loss 4.4842
[PyTorch] epoch 2/15, loss 4.4452
[PyTorch] epoch 3/15, loss 4.3533
[PyTorch] epoch 4/15, loss 4.3379
[PyTorch] epoch 5/15, loss 4.3062
[PyTorch] epoch 6/15, loss 4.2859
[PyTorch] epoch 7/15, loss 4.2311
[PyTorch] epoch 8/15, loss 4.1350
[PyTorch] epoch 9/15, loss 3.9356
[PyTorch] epoch 10/15, loss 3.7844
[

In [181]:
compare_models(
    w2v_models, X_train, Y_train, X_test, Y_test,
    embedding=word_vectors, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual             0.0463   0.0200
  pytorch            0.5537   0.1200
  tensorflow         0.5244   0.1500

=== generation from 'Machine' ===
  manual           Machine data vehicles. applications programmed. (AI) and identify healthcare a without
  pytorch          Machine those diagnostics, learning, make speech helping organizations Intelligence such as
  tensorflow       Machine supervised supervised is branch of make types of of or


### 3.2 GloVe — pre-trained embeddings (Google-News word2vec)

Swap our small in-house vectors for Google's pre-trained 300-d word2vec vectors (trained on ~100B words of Google News).

In [182]:
# pre-trained vectors from the Google-News dataset (300d, 3M words)
import gensim.downloader as api
glove = api.load("word2vec-google-news-300")

In [183]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, word_vectors=glove)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 72 training sequences of length 5 from a corpus of 78 words, with a vocabulary of 67 unique words.
X: (72, 5, 300)   Y: (72, 5, 67)   (m, T_x, vector_size)
Split 72 sequences -> 58 train / 14 test.


In [184]:
hidden_layers, lr, epochs, batch_size, task = (100,), (8.0, 0.03, 0.03), 15, 32, 'classification'
print('GloVe Models')
glove_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

GloVe Models
----------------------------------------------------------------------------------------------------
Training Manual LSTM
epoch 1/15 - loss: 4.2017
epoch 2/15 - loss: 3.9825
epoch 3/15 - loss: 3.7183
epoch 4/15 - loss: 3.3763
epoch 5/15 - loss: 2.8097
epoch 6/15 - loss: 2.2265
epoch 7/15 - loss: 1.7422
epoch 8/15 - loss: 1.4680
epoch 9/15 - loss: 1.0831
epoch 10/15 - loss: 0.7784
epoch 11/15 - loss: 0.5976
epoch 12/15 - loss: 0.4773
epoch 13/15 - loss: 0.4182
epoch 14/15 - loss: 0.3120
epoch 15/15 - loss: 0.2548

----------------------------------------------------------------------------------------------------
Training PyTorch LSTM
[PyTorch] epoch 1/15, loss 4.1024
[PyTorch] epoch 2/15, loss 3.0177
[PyTorch] epoch 3/15, loss 1.4417
[PyTorch] epoch 4/15, loss 0.5514
[PyTorch] epoch 5/15, loss 0.2384
[PyTorch] epoch 6/15, loss 0.1260
[PyTorch] epoch 7/15, loss 0.0795
[PyTorch] epoch 8/15, loss 0.0564
[PyTorch] epoch 9/15, loss 0.0468
[PyTorch] epoch 10/15, loss 0.0458
[PyT

In [185]:
compare_models(
    glove_models, X_train, Y_train, X_test, Y_test,
    embedding=glove, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual             0.9690   0.9286
  pytorch            0.9759   0.9714
  tensorflow         0.9759   0.9714

=== generation from 'Machine' ===
  manual           Machine learning is widely used in various applications such as as
  pytorch          Machine Learning is branch Artificial Intelligence that enables computers learn from
  tensorflow       Machine Learning is branch Artificial Intelligence that enables computers computers learn


### 3.3 FastText embeddings

FastText builds word vectors from character n-grams, so it can embed even rare or out-of-vocabulary words. Trained here on our own sentences.

In [186]:
from gensim.models import FastText

ft_model = FastText(
    sentence_words,
    vector_size=150,  # dimensionality of the word vectors
    window=5,        # context window size
    min_count=1,     # minimum word frequency to include in the model
    workers=4,       # number of CPU cores to use for training
    sg=1             # use skip-gram architecture (sg=0 for CBOW)
)
fasttext_vectors = ft_model.wv

In [187]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, word_vectors=fasttext_vectors)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 words, with a vocabulary of 88 unique words.
X: (102, 5, 150)   Y: (102, 5, 88)   (m, T_x, vector_size)
Split 102 sequences -> 82 train / 20 test.


In [188]:

hidden_layers, lr, epochs, batch_size, task = (50,), (8.0, 0.03, 0.03), 15, 32, 'classification'
print('FastText Models')
fasttext_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

FastText Models
----------------------------------------------------------------------------------------------------
Training Manual LSTM
epoch 1/15 - loss: 4.4602
epoch 2/15 - loss: 4.4125
epoch 3/15 - loss: 4.3753
epoch 4/15 - loss: 4.3838
epoch 5/15 - loss: 4.3789
epoch 6/15 - loss: 4.3711
epoch 7/15 - loss: 4.3729
epoch 8/15 - loss: 4.3606
epoch 9/15 - loss: 4.3613
epoch 10/15 - loss: 4.3553
epoch 11/15 - loss: 4.3652
epoch 12/15 - loss: 4.3764
epoch 13/15 - loss: 4.3642
epoch 14/15 - loss: 4.3513
epoch 15/15 - loss: 4.3443

----------------------------------------------------------------------------------------------------
Training PyTorch LSTM
[PyTorch] epoch 1/15, loss 4.4776
[PyTorch] epoch 2/15, loss 4.4069
[PyTorch] epoch 3/15, loss 4.3579
[PyTorch] epoch 4/15, loss 4.3252
[PyTorch] epoch 5/15, loss 4.3411
[PyTorch] epoch 6/15, loss 4.3515
[PyTorch] epoch 7/15, loss 4.3301
[PyTorch] epoch 8/15, loss 4.3381
[PyTorch] epoch 9/15, loss 4.3314
[PyTorch] epoch 10/15, loss 4.3226
[

In [189]:
compare_models(
    fasttext_models, X_train, Y_train, X_test, Y_test,
    embedding=fasttext_vectors, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual             0.0463   0.0400
  pytorch            0.0537   0.0300
  tensorflow         0.0756   0.0400

=== generation from 'Machine' ===
  manual           Machine Machine patterns role widely amount and recommendation an increasingly in
  pytorch          Machine ML processing, branch without vehicles. healthcare data-driven continues The widely
  tensorflow       Machine Machine patterns applications in available learning machine organizations Intelligence use


### 3.4 BERT embeddings

Use a pre-trained BERT model to produce a contextual `[CLS]` embedding per word, then feed those 768-d vectors to the LSTMs.

In [190]:
from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

bert_word_to_vec = {}
for word in words:
    inputs = tokenizer(word, return_tensors="pt")
    with torch.no_grad():
        outputs = bert_model(**inputs)
    bert_word_to_vec[word] = outputs.last_hidden_state[:, 0, :].squeeze().numpy()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [191]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, word_vectors=bert_word_to_vec)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")
X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 words, with a vocabulary of 88 unique words.
X: (102, 5, 768)   Y: (102, 5, 88)   (m, T_x, vector_size)
Split 102 sequences -> 82 train / 20 test.


In [192]:
hidden_layers, lr, epochs, batch_size, task = (100,), (8.0, 0.03, 0.03), 15, 32, 'classification'
print('BERT Models')
bert_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

BERT Models
----------------------------------------------------------------------------------------------------
Training Manual LSTM
epoch 1/15 - loss: 4.7091
epoch 2/15 - loss: 5.7656
epoch 3/15 - loss: 4.6002
epoch 4/15 - loss: 4.4776
epoch 5/15 - loss: 4.4349
epoch 6/15 - loss: 4.3545
epoch 7/15 - loss: 4.3562
epoch 8/15 - loss: 4.2991
epoch 9/15 - loss: 4.3195
epoch 10/15 - loss: 4.2833
epoch 11/15 - loss: 4.3030
epoch 12/15 - loss: 4.3141
epoch 13/15 - loss: 4.2295
epoch 14/15 - loss: 4.3604
epoch 15/15 - loss: 4.3064

----------------------------------------------------------------------------------------------------
Training PyTorch LSTM
[PyTorch] epoch 1/15, loss 4.5373
[PyTorch] epoch 2/15, loss 4.3256
[PyTorch] epoch 3/15, loss 3.9983
[PyTorch] epoch 4/15, loss 3.7108
[PyTorch] epoch 5/15, loss 3.3443
[PyTorch] epoch 6/15, loss 2.9960
[PyTorch] epoch 7/15, loss 2.6269
[PyTorch] epoch 8/15, loss 2.2784
[PyTorch] epoch 9/15, loss 1.9467
[PyTorch] epoch 10/15, loss 1.6242
[PyTo

In [193]:
compare_models(
    bert_models, X_train, Y_train, X_test, Y_test,
    embedding=bert_word_to_vec, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual             0.0610   0.0300
  pytorch            0.8976   0.7900
  tensorflow         0.9220   0.8100

=== generation from 'Machine' ===
  manual           Machine to is being systems, and available machine important use The
  pytorch          Machine learning are supervised learning, unsupervised learning, unsupervised learning, and reinforcement
  tensorflow       Machine learning is widely various applications and improve their performance without
